# Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Reading from Bronze

In [0]:
df = spark.table("data_engineering_2026.bronze.crm_cust_info")

# Data Transformation

## 1.Rename the columns

In [0]:
rename_cols = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status":"martial_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

for old_col, new_col in rename_cols.items():
    df = df.withColumnRenamed(old_col, new_col)

## 2. Trim the Columns

In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## 3. Normalization

In [0]:
df = df.withColumn("martial_Status",
    when(col("martial_status") == "M", "Married")
    .when(col("martial_status") == "S", "Single")
    .otherwise("Unknown")
)\
.withColumn("gender",
    when(col("gender") == "M", "Male")
    .when(col("gender") == "F", "Female")
    .otherwise("Unknown")
)\
.withColumn("create_date",
    when(col("create_date").isNull(), to_date(lit("1900-01-01")))
    .otherwise(col("create_date")))

df = df.withColumn("first_name",
    when(col("first_name").isNull() | (trim(col("first_name")) == ""), "Unknown")
    .otherwise(trim(col("first_name")))
)\
.withColumn("last_name",
    when(col("last_name").isNull() | (trim(col("last_name")) == ""), "Unknown")
    .otherwise(trim(col("last_name"))))

df = df.withColumn("full_name",
    when((col("first_name") != "Unknown") & (col("last_name") != "Unknown"),
    concat(col("first_name"), lit(" "), col("last_name"))
    ).when(
        col("first_name") != "Unknown",
        col("first_name")
    ).when(
        col("last_name") != "Unknown",
        col("last_name")
    ).otherwise("Unknown")
)

In [0]:
df = df.filter(
    col("customer_id").isNotNull() & (trim(col("customer_id")) != ""))

In [0]:
df = df.dropDuplicates(["customer_id"])

In [0]:
df.display()

In [0]:
df.display()

In [0]:
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

# Writing in Silver

In [0]:
df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("data_engineering_2026.silver.Silver_crm_customers")

In [0]:
df.display()

In [0]:
prd = spark.table("data_engineering_2026.bronze.crm_prd_info")

## Renaming the columns

In [0]:
Rename_col_prd = {
    "prd_id":"product_id",
    "prd_key":"product_number",
    "prd_nm":"product_name",
    "prd_cost": "product_cost",
    "prd_line":"product_line",
    "prd_start_dt":"start_date",
    "prd_end_dt":"end_date"
}

for old_col, new_col in Rename_col_prd.items():
    prd = prd.withColumnRenamed(old_col, new_col)

prd.display()

## Triming the Columns

In [0]:
for field in prd.schema.fields:
    if isinstance(field.dataType, StringType):
        prd = prd.withColumn(field.name, trim(col(field.name)))

prd.display()


In [0]:
parts = split(col("product_number"), "-")

prd = prd.withColumn(
    "category_id",
    concat_ws("-", parts[0], parts[1])
)

In [0]:
prd.display()

In [0]:
prd = prd.withColumn("product_cost", coalesce(col("product_cost"), lit(0)))

## Normalization

In [0]:
prd = prd.withColumn("product_line",
    when(upper(col("product_line")) == "M", "Mountain")
    .when(upper(col("product_line")) == "R", "Road")
    .when(upper(col("product_line")) == "T", "Touring")
    .when(upper(col("product_line")) == "S", "Other_Sales")
    .otherwise("Not available")
)

In [0]:
prd.display()

In [0]:
from pyspark.sql.functions import col, when, lit, to_date

prd = prd.withColumn(
    "end_date",
    when(col("end_date").isNull(), to_date(lit("9999-12-31")))
    .otherwise(col("end_date"))
)

In [0]:
prd = prd.withColumn(
    "product_number",
    regexp_extract(col("product_number"), r'^[^-]+-[^-]+-(.*)', 1)
)

In [0]:
prd.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in prd.columns
]).show()

In [0]:
prd.display()

In [0]:
(
prd.write
	.mode("overwrite")
	.format("delta")\
     .option("overwriteSchema", "true")\
	.saveAsTable("data_engineering_2026.silver.Silver_crm_products")
)

# Reading Sales_Details Table from Bronze

In [0]:
sales_df = spark.read.table("data_engineering_2026.bronze.crm_sales_details")

sales_df.display()

## Trimming the Sales_Detail Table

In [0]:
for field in sales_df.schema.fields:
  if isinstance(field.dataType, StringType):
	  sales_df = sales_df.withColumn(field.name, trim(col(field.name)))


In [0]:
Rename_cols = {
	"sls_ord_num": "order_number",
	"sls_prd_key": "product_number",
	"sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
	"sls_ship_dt": "ship_date",
	"sls_due_dt": "due_date",
	"sls_sales": "sales_amount",
	"sls_quantity": "quantity",
	"sls_price": "price",
}

for old_name, new_name in Rename_cols.items():
	sales_df = sales_df.withColumnRenamed(old_name, new_name)


In [0]:
sales_df = (
	sales_df.withColumn("order_date",
		when(
			(col("order_date") == 0) | (length(col("order_date").cast("string")) !=8), lit(None))
		.otherwise(to_date(col("order_date").cast("string"), "yyyyMMdd"))
		))\
        .withColumn("ship_date",
		when( 
			(col("ship_date") == 0) | (length(col("ship_date")) != 8), lit(None))
		.otherwise(to_date(col("ship_date").cast("string"), "yyyyMMdd"))
		)\
		.withColumn("due_date",
		when(
			(col("due_date") == 0) | (length(col("due_date")) !=8), lit(None))
		.otherwise(to_date(col("due_date").cast("string"), "yyyyMMdd"))
		)

In [0]:
sales_df = (sales_df
        .withColumn("sales_amount",
            when(col("sales_amount").isNull(),0)\
            .otherwise(col("sales_amount"))
    )
    .withColumn("quantity",
            when(col("quantity").isNull(),0)\
            .otherwise(col("quantity"))
    ))


In [0]:
sales_df = sales_df.withColumn("order_date",
    when(col("order_date").isNull(), to_date(lit("1900-01-01")))
    .otherwise(col("order_date"))
)\
.withColumn("price",\
    col("sales_amount") * col("quantity"))


In [0]:
sales_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in sales_df.columns
]).show()

In [0]:
sales_df = sales_df.filter(col("sales_amount") >= 0)

In [0]:
sales_df.display()

## Writing Sales_Details in Silver

In [0]:
sales_df.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("data_engineering_2026.silver.silver_crm_sales")

In [0]:
sales_df.display()

In [0]:
sales_df.display()